In [1]:
import pandas as pd 
import numpy as np
import random
import os 
import argparse
import json
import torch
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import TensorDataset
from attrdict import AttrDict
from transformers import BertConfig, BertTokenizer, BertModel
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertConfig, DistilBertForSequenceClassification
from transformers import RobertaModel, RobertaTokenizer, RobertaConfig, RobertaForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup

In [2]:
default_path = os.getcwd()
data_path = os.path.join(default_path, '../data/BWS/score')
base_model = os.path.join(default_path, '../base-model')
config_path = os.path.join(default_path, '../config')
log_path = os.path.join(default_path, '../log')
model_path = os.path.join(default_path, '../models/pseudo')
config_file = "bert-base.json"

In [416]:
criteria = 4
seed = 42
fold = 4

In [417]:
a1_fold_train = pd.read_csv(os.path.join(data_path, f'a{criteria}_fold{fold}_train.csv'))
a1_fold_test = pd.read_csv(os.path.join(data_path, f'a{criteria}_fold{fold}_test.csv'))

In [418]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

In [419]:
training_config.seed

42

In [420]:
training_config.seed = seed

In [421]:
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

In [422]:
training_config.config_path = config_path
training_config.log_path = log_path
training_config.data_path = data_path
training_config.model_path = model_path 

In [423]:
model_name = 'roberta-base'   # RobertaModel

In [424]:
tokenizer = RobertaTokenizer.from_pretrained(model_name, model_max_length=128)
config = RobertaConfig.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name, config=config)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [425]:
training_config.pad = 'max_length'
training_config.num_epochs  = 10

In [426]:
class DSMDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [427]:
class DSMProcessor():
    def __init__(self, config, training_config, tokenizer, truncation=True):
        self.tokenizer = tokenizer 
        self.max_len =  config.max_position_embeddings
        self.pad = training_config.pad
        self.batch_size = training_config.train_batch_size
        self.truncation = truncation
    
    def convert_data(self, data_file):
        context2 = None    # single sentence classification
        batch_encoding = self.tokenizer.batch_encode_plus(
            [(data_file[idx][0], context2) for idx in range(len(data_file))],   # text, 
            max_length = self.max_len,
            padding = self.pad,
            truncation = self.truncation
        )
        
        features = []
        for i in range(len(data_file)):
            inputs = {k: batch_encoding[k][i] for k in batch_encoding}
            try:
                inputs['label'] = data_file[i][1] 
            except:
                inputs['label'] = 0 
            # print(inputs)
            features.append(inputs)
        
        all_input_ids = torch.tensor([f['input_ids'] for f in features], dtype=torch.long)
        all_attention_mask = torch.tensor([f['attention_mask'] for f in features], dtype=torch.long)
        # all_token_type_ids = torch.tensor([f['token_type_ids'] for f in features], dtype=torch.long)
        all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)

        dataset = TensorDataset(all_input_ids, all_attention_mask, all_labels)
        return dataset
    
    def shuffle_data(self, dataset, data_type):
        if data_type == 'train':
            return RandomSampler(dataset)
        elif data_type == 'eval' or data_type == 'test':
            return SequentialSampler(dataset)
        
    def load_data(self, dataset, sampler):
        return DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

In [428]:
training_config.save_model_path = model_path 
training_config.log_path = log_path

In [429]:
config.max_position_embeddings = 128
config.max_position_embeddings

128

In [430]:
class BertRegressor(nn.Module):
    def __init__(self, config, model):
        super(BertRegressor, self).__init__()
        self.model = model
        self.linear = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.last_hidden_state[:, 0, :]
        # print(f'logits: {len(logits)}, {len(logits[0])}')
        x = self.linear(logits)
        x = self.relu(x)
        score = self.out(x)
        # print(f'score: {score}')
        return score 

In [431]:
def RMSELoss(yhat,y):
    return torch.sqrt(torch.mean((yhat-y)**2))

In [432]:
class BertTrainer():
    def __init__(self, config, training_config, model, model_name, train_dataloader, eval_dataloader):
        self.config = config
        self.training_config = training_config
        self.model = model
        self.model_name = model_name
        self.train_dataloader = train_dataloader
        self.eval_dataloader = eval_dataloader
        
    def set_seed(self):
        random.seed(self.training_config.seed)
        np.random.seed(self.training_config.seed)
        torch.manual_seed(self.training_config.seed)
        if not self.training_config.no_cuda and torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.training_config.seed)
    
    def train(self):
        global_step = 0; nb_eval_steps = 0
        train_rmse = []; eval_rmse = []
        t_total = len(self.train_dataloader) // self.training_config.gradient_accumulation_steps * self.training_config.num_epochs

        optimizer = AdamW(self.model.parameters(), lr=self.training_config.learning_rate, eps=self.training_config.adam_epsilon)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(t_total * self.training_config.warmup_proportion), \
                                                    num_training_steps=t_total)
        
        criterion = RMSELoss
        # criterion = nn.MSELoss()
        best_loss = 9999 
        
        self.model.zero_grad()
        for epoch in range(int(self.training_config.num_epochs)):
            train_loss = 0.0; eval_loss = 0.0 
            
            for step, batch in enumerate(self.train_dataloader):
                self.model.train()
                batch = tuple(t.to(self.training_config.device) for t in batch)
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    # "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                # print(f'output: {type(outputs)}, {outputs.squeeze}')
                label = batch[2]
                # print(f'label: {label}')
                # print(f'output: {outputs}, {outputs.squeeze()}')
                loss = criterion(outputs.squeeze(), batch[2].type_as(outputs))
                loss.backward()
                
                train_loss += loss.item()
                # torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.training_config.max_grad_norm)
                optimizer.step()
                scheduler.step()
                
                self.model.zero_grad()
            
            print(f'epoch: {epoch + 1} done, train_loss: {train_loss / len(self.train_dataloader)}')
            train_rmse.append(train_loss / len(self.train_dataloader))

            for step2, batch2 in enumerate(self.eval_dataloader):
                self.model.eval()
                batch2 = tuple(t.to(self.training_config.device) for t in batch2)

                with torch.no_grad():
                    inputs = {
                        "input_ids": batch2[0],
                        "attention_mask": batch2[1],
                        # "token_type_ids": batch2[2],
                    }
                    label2 = batch2[2]
                    outputs = self.model(**inputs)
                    tmp_eval_loss = criterion(outputs.squeeze(), label2.type_as(outputs))
                    eval_loss += tmp_eval_loss.mean().item()
                    
                nb_eval_steps += 1

            eval_loss = eval_loss / nb_eval_steps
            eval_rmse.append(eval_loss)

        # self.save_model(os.path.join(self.training_config.model_path, self.model_name, f'bert_bws_5.pt'))
        # self.save_log(train_rmse, eval_rmse, epoch+1)
        return train_rmse, eval_rmse
            
    def save_log(self, train_mse, eval_mse, epoch):
        with open(os.path.join(self.training_config.log_path, self.model_name, f'train_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(train_mse, f, pickle.HIGHEST_PROTOCOL)  
        
        with open(os.path.join(self.training_config.log_path, self.model_name, f'eval_{epoch}_mse.pickle'), 'wb') as f:
            pickle.dump(eval_mse, f, pickle.HIGHEST_PROTOCOL)  
    
    def save_model(self, model_name):
        torch.save(self.model.state_dict(), model_name)

In [433]:
training_config.seed

42

In [434]:
dsm_processor = DSMProcessor(config, training_config, tokenizer)

In [435]:
train_file = DSMDataset(a1_fold_train)
val_file = DSMDataset(a1_fold_test)

In [436]:
train_dataset = dsm_processor.convert_data(train_file)
val_dataset = dsm_processor.convert_data(val_file)
# test_dataset = dsm_processor.convert_data(test_file)

In [437]:
train_sampler = dsm_processor.shuffle_data(train_dataset, 'train')
val_sampler = dsm_processor.shuffle_data(val_dataset, 'eval')
# test_sampler = dsm_processor.shuffle_data(test_dataset, 'test')

In [438]:
train_dataloader = dsm_processor.load_data(train_dataset, train_sampler)
val_dataloader = dsm_processor.load_data(val_dataset, val_sampler)
# test_dataloader = dsm_processor.load_data(test_dataset, test_sampler)

In [439]:
model_reg = BertRegressor(config, model).to(training_config.device)  

In [440]:
distil_model = f'roberta_pseudo_a{criteria}_s{seed}_e5.pt'

In [441]:
model_reg.load_state_dict(torch.load(os.path.join(model_path, distil_model)))
model_reg.to(training_config.device)

BertRegressor(
  (model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0): RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,

In [442]:
bert_trainer = BertTrainer(config, training_config, model_reg, model_name, train_dataloader, val_dataloader)

In [443]:
train_mse, eval_mse = bert_trainer.train()

C:\Users\lamda\.conda\envs\aaai\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


epoch: 1 done, train_loss: 2.093980503578981
epoch: 2 done, train_loss: 1.6016087368130685
epoch: 3 done, train_loss: 1.22856351484855
epoch: 4 done, train_loss: 1.0568121495346228
epoch: 5 done, train_loss: 0.9181547408302625
epoch: 6 done, train_loss: 0.7688061264654
epoch: 7 done, train_loss: 0.657348861048619
epoch: 8 done, train_loss: 0.5468694710483154
epoch: 9 done, train_loss: 0.49226435596744217
epoch: 10 done, train_loss: 0.42514625899493697


In [444]:
bert_trainer.save_model(os.path.join(model_path, f'roberta_pseudo_a{criteria}_s{seed}_fold{fold}_e10.pt'))

In [445]:
class BertRegTester():
    def __init__(self, training_config, model):
        self.training_config = training_config
        self.model = model

    def get_label(self, test_dataloader, test_type):
        '''
        test_type: 0  -> Test dataset 
        test_type: 1  -> Test sentence
        '''
        preds = []
        labels = []

        for batch in test_dataloader:
            self.model.eval()   # self 안 붙이면 이상한 Output (BaseModelOutputWithPoolingAndCrossAttentions) 출력 
            batch = tuple(t.to(self.training_config.device) for t in batch)   # args.device: cuda 
            with torch.no_grad():
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    # "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                if test_type == 0:
                    preds.extend(outputs.squeeze().detach().cpu().numpy())
                elif test_type == 1:
                    preds.extend(outputs[0].detach().cpu().numpy())            
            label = batch[2].detach().cpu().numpy()
            labels.extend(label)
        return preds, labels 

In [446]:
bws_tester = BertRegTester(training_config, model_reg)

In [447]:
bws_pred, bws_true = bws_tester.get_label(val_dataloader, 0)

In [448]:
bws_pred[100:103], bws_true[100:103]

([8.418107, 5.322442, 7.167931], [7, 7, 6])

In [449]:
from torchmetrics import R2Score

criterion = RMSELoss
r2score = R2Score()
bws_rmse = criterion(torch.Tensor(bws_pred), torch.Tensor(bws_true))
bws_r2 = r2score(torch.Tensor(bws_pred), torch.Tensor(bws_true))
bws_rmse, bws_r2

(tensor(2.0633), tensor(0.4086))

In [450]:
len(bws_pred)

320

In [451]:
bws_pred = pd.DataFrame(bws_pred, columns=['pred'])
bws_pred.head(3)

,pred
0,5.637151
1,2.992970
2,6.650236


In [452]:
bws_pred.to_csv(os.path.join(data_path, f'roberta_pseudo_a{criteria}_s{seed}_fold{fold}_pred.csv'), index=False)

In [453]:
# 42 -> 21 -> 3    rev
# 1 -> 2 -> 3 -> 4    rev  